# Лабораторная работа №3: Стохастическое моделирование. Методы Монте-Карло.

**Тема:** Моделирование идеального газа в ящике и проверка распределения Максвелла.  
**Дисциплина:** Компьютерное моделирование (СМ6, МГТУ им. Н.Э. Баумана).  
**Преподаватель:** Федулов Вадим Алексеевич.  

## 1. Введение и цель работы

**Цель работы:** Освоить метод статистического моделирования (Монте-Карло) на примере системы многих частиц. Реализовать модель идеального газа в двумерном ящике, исследовать процесс установления термодинамического равновесия и проверить выполнение закона распределения скоростей Максвелла-Больцмана.

**Актуальность:** 
Данная работа напрямую связана с материалом **Лекции №6 «Методы динамики частиц»**. В лекции рассматривались подходы к моделированию молекулярной динамики, потенциалы взаимодействия (Леннарда-Джонса), граничные условия и свойства переноса (диффузия, равновесие). 
В данной лабораторной работе мы используем упрощенную модель (идеальный газ без межчастичного потенциала), чтобы сосредоточиться на статистических аспектах метода Монте-Карло и визуализации эволюции системы во времени.

## 2. Математическая модель

### 2.1. Физическая постановка
Рассматривается система из $N$ идентичных точечных частиц массой $m$, заключенных в двумерный квадратный ящик размером $L \times L$. 
*   **Взаимодействие:** Частицы не взаимодействуют друг с другом (модель идеального газа), но испытывают абсолютно упругие столкновения со стенками ящика.
*   **Термостат:** Температура $T$ задается через начальное распределение скоростей.

### 2.2. Алгоритм движения (Метод частиц)
Эволюция системы дискретна по времени. Пусть $\vec{r}_i(t) = (x_i, y_i)$ — координаты, а $\vec{v}_i(t) = (v_{xi}, v_{yi})$ — скорости $i$-й частицы в момент времени $t$. Шаг по времени $\Delta t = 1$ (в безразмерных единицах).

1.  **Перемещение (Free streaming):**
    $$ \vec{r}_i(t + \Delta t) = \vec{r}_i(t) + \vec{v}_i(t) \cdot \Delta t $$

2.  **Граничные условия (Упругое отражение):**
    Если частица выходит за пределы ящика $[0, L]$, происходит инверсия нормальной компоненты скорости и возврат координаты внутрь области (зеркальное отражение).
    
    Для координаты $x$:
    $$
    \text{Если } x_i < 0: \quad v_{xi} \leftarrow -v_{xi}, \quad x_i \leftarrow -x_i
    $$
    $$
    \text{Если } x_i > L: \quad v_{xi} \leftarrow -v_{xi}, \quad x_i \leftarrow 2L - x_i
    $$
    Аналогично для координаты $y$. Это обеспечивает сохранение энергии в системе.

### 2.3. Статистика и Распределение Максвелла
Согласно молекулярно-кинетической теории, в состоянии равновесия модули скоростей частиц в 2D-газе подчиняются **распределению Рэлея** (частный случай распределения Максвелла для двумерного случая).

Плотность вероятности $f(v)$ иметь скорость $v = |\vec{v}|$:
$$ f(v) = \frac{m}{k_B T} v \exp\left( -\frac{m v^2}{2 k_B T} \right) $$

В нашей модели принята безразмерная система единиц, где $m=1, k_B=1$. Параметр `TEMPERATURE` в коде соответствует величине $T$ (дисперсии распределения компонент скоростей). Теоретическая кривая для сравнения:
$$ f(v) = \frac{v}{T} \exp\left( -\frac{v^2}{2 T} \right) $$

Средняя кинетическая энергия одной частицы в 2D случае связана с температурой соотношением:
$$ \langle E_k \rangle = \frac{1}{2} m \langle v^2 \rangle = k_B T $$
В наших единицах: $\langle E_k \rangle \approx T$.

## 3. Ход работы и программная реализация

Работа выполнялась итеративно, с постепенным усложнением модели.

### Этап 1: Базовая визуализация динамики
**Файл:** `gas_simulation_part1.py`

На первом этапе была реализована базовая симуляция движения частиц.
*   **Инициализация:** Координаты задавались равномерно случайно (`np.random.rand`), скорости — согласно нормальному распределению (`np.random.normal`) с дисперсией, равной температуре.
*   **Визуализация:** Использовалась библиотека `matplotlib.animation`. Цвет каждой частицы кодировался её текущей скоростью (colormap `jet`), что позволяло визуально отслеживать "горячие" (быстрые) и "холодные" (медленные) зоны.

**Результат:** Получена анимация хаотического движения частиц в замкнутом объеме.
![Анимация газа](gas_box_animation.gif)

### Этап 2: Статистический анализ (Метод Монте-Карло)
**Файл:** `gas_simulation_part2.py`

Здесь мы перешли от визуализации к сбору данных. Суть метода Монте-Карло в данном контексте — набор большой выборки измерений для построения статистического распределения.
*   Увеличено число частиц до $N=500$.
*   Количество шагов симуляции: 1000.
*   Сбор данных: На каждом 10-м шаге скорости всех частиц записывались в общий массив `all_speeds`.
*   Построена гистограмма плотностей вероятности и наложена теоретическая кривая Максвелла.

**Результаты консоли:**
> Собрано 50000 измерений скорости.  
> Средняя кинетическая энергия: 1.471 (при заданной T=1.5).

Отклонение средней энергии от заданной температуры составило менее 2%, что подтверждает корректность работы численной схемы и сохранение энергии.

![Распределение Максвелла](maxwell_distribution.png)

### Этап 3: Динамическая визуализация сходимости
**Файл:** `gas_simulation_final_fixed.py`

Для демонстрации принципа сходимости метода Монте-Карло был создан объединенный скрипт.
*   **Левая панель:** Движение частиц в реальном времени.
*   **Правая панель:** Гистограмма распределения скоростей, которая строится "на лету".
*   **Особенность реализации:** Использован метод `ax.clear()` на каждом кадре для корректного обновления гистограммы.

На анимации видно, как при малом числе измерений (начало GIF) гистограмма имеет шумный, неровный характер, но по мере накопления статистики (к концу GIF) она плавно принимает форму теоретической красной кривой.

![Финальная анимация сходимости](final_gas_report.gif)

### Этап 4: Эксперимент "Смешивание газов" (Необязательный/Дополнительный)
**Файл:** `gas_experiment_mixing.py`

Проведен численный эксперимент, иллюстрирующий второе начало термодинамики (выравнивание температур).
1.  Ящик разделен перегородкой на две части.
2.  Слева — "горячий" газ ($T=3.0$), справа — "холодный" ($T=0.5$).
3.  На кадре №100 перегородка убирается.
4.  Наблюдается диффузия и перемешивание. На графике справа видно, как два раздельных пика распределения скоростей сливаются в одно общее равновесное распределение.

![Смешивание газов](gas_mixing_experiment.gif)

## 4. Выводы

В ходе лабораторной работы был успешно освоен метод Монте-Карло применительно к задачам статистической физики.

1.  **Реализация модели:** Разработано программное обеспечение на языке Python, моделирующее поведение ансамбля частиц в замкнутом объеме с использованием алгоритмов молекулярной динамики (свободный полет + упругое отражение).
2.  **Верификация:** Численно подтверждено, что система приходит в состояние статистического равновесия. Распределение модулей скоростей, полученное в результате симуляции, с высокой точностью совпадает с теоретическим распределением Максвелла-Больцмана для двумерного газа.
3.  **Энергетический баланс:** Показано, что средняя кинетическая энергия частиц в установившемся режиме соответствует заданной температуре термостата (погрешность ~2%).
4.  **Визуализация:** Продемонстрирована эволюция статистического ансамбля. Наглядно показано, как закон больших чисел работает в методе Монте-Карло: флуктуации уменьшаются, и экспериментальная гистограмма сходится к теоретической плотности вероятности при увеличении объема выборки.
5.  **Связь с курсом:** Работа демонстрирует практическое применение методов динамики частиц (Лекция 6) для исследования фундаментальных свойств материи (равновесие, свойства переноса), даже в упрощенной постановке без учета парных потенциалов взаимодействия.

---
*Работа выполнена с использованием библиотек `numpy`, `matplotlib`, `scipy`.*